# Synapse: The Token Budget Lab

Welcome to the hands-on lab for **Phase 3: Tokens, Context, And Prompt Budgets**.

In the sessions leading up to this lab, you learned that every AI feature has a **token shape** — a distinct mix of input, output, context, and schema tokens that drives its latency and cost.

### 🎯 The Scenario

You are engineering an AI-powered support assistant. It has three features:

1. **Summarize** a support transcript before a human agent takes over
2. **Extract** invoice fields into JSON for the billing system
3. **Answer** a policy question with retrieved context (RAG-style)

The product is live. The PM says it feels slow and a recent invoice flagged the AI cost at **4x the target**. You have no token logs. Nobody measured anything.

### 🗺️ What You Will Do

1. Make the three feature calls and read the token bill
2. Build a budget worksheet — dominant bucket, latency risk, cost risk
3. Replicate the expensive trace and find the caching opportunity
4. Add token metadata to an eval record so the next failure is debuggable

## 🛠️ Step 0: Setup

You need an **OpenAI API key**. The cell below installs dependencies and prompts for your key.

In [ ]:
import os, time, json, getpass
from typing import Optional
from dataclasses import dataclass, asdict
from pydantic import BaseModel, Field

try:
    from openai import OpenAI
except ImportError:
    !pip install -q openai pydantic
    from openai import OpenAI

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Paste your OpenAI API key: ')

client = OpenAI()
print('✅ Ready')

## 📊 Step 1: Three Calls — No Budget, No Logs

Below are the three support-assistant features. Each makes a real API call.

Right now, **none of them logs tokens**. Run them and notice: you get an answer back, but you have no idea what it cost or why it was slow.

In [ ]:
TRANSCRIPT = """
Agent: Hi, thanks for calling SynapseCloud support. How can I help?
Customer: I can't access my billing dashboard. It says 'permission denied'.
Agent: Let me check your account. Can I get your email?
Customer: Sure, it's alex@company.io.
Agent: I see the issue — your billing role was accidentally removed in last week's migration.
Customer: That's frustrating. How long will the fix take?
Agent: I'm adding the role now. You should have access in about two minutes.
Customer: Okay, I'll wait. Also, can you email me a receipt for last month?
Agent: Absolutely. I'll send that right now. Is there anything else?
Customer: No, that's everything. Thanks.
Agent: You're welcome. Have a great day!
"""

INVOICE_TEXT = """
Invoice #INV-2024-0891
Date: November 14, 2024
Bill To: Acme Corp, 123 Main St, San Francisco CA 94102
Line Items:
  - SynapseCloud Pro (annual): $1,200.00
  - Overage: 50,000 extra API calls @ $0.001: $50.00
Subtotal: $1,250.00
Tax (8.5%): $106.25
Total Due: $1,356.25
Due Date: December 14, 2024
"""

POLICY_CHUNKS = """
[Policy: Refund Eligibility]
Annual subscriptions are eligible for a full refund within 30 days of purchase.
After 30 days, refunds are prorated by months remaining.
Overage charges are non-refundable.

[Policy: Cancellation]
Customers may cancel at any time. Service continues until the end of the billing period.
Cancellation requests must be submitted via the billing dashboard or by contacting support.
"""

# ── Feature 1: Summarize ──────────────────────────────────────────────────────
def summarize_transcript(transcript: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Summarize this support transcript in 2-3 sentences for the handoff agent.'},
            {'role': 'user', 'content': transcript},
        ],
    )
    return resp.choices[0].message.content

# ── Feature 2: Extract ────────────────────────────────────────────────────────
def extract_invoice(invoice_text: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Extract invoice fields as JSON: invoice_number, date, customer_name, subtotal, tax, total, due_date.'},
            {'role': 'user', 'content': invoice_text},
        ],
    )
    return resp.choices[0].message.content

# ── Feature 3: RAG-style Answer ───────────────────────────────────────────────
def answer_policy_question(question: str, context: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Answer the customer question using only the provided policy context. Cite the policy section.'},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content

print('Running three unbudgeted calls...\n')
s = summarize_transcript(TRANSCRIPT)
e = extract_invoice(INVOICE_TEXT)
a = answer_policy_question('Can I get a refund after 45 days?', POLICY_CHUNKS)

print('Summarize output:', s[:120], '...')
print('Extract output:  ', e[:120], '...')
print('Answer output:   ', a[:120], '...')
print('\n❓ How much did that cost? How long did each call take? We have no idea.')

## 💸 Step 2: Read the Token Bill

Every OpenAI response includes a `usage` object: `prompt_tokens`, `completion_tokens`, `total_tokens`.

**Your task:** Rewrite each function to return both the answer AND the usage. Then print a token report for all three calls.

Fill in the `# --- YOUR CODE HERE ---` blocks.

In [ ]:
@dataclass
class CallResult:
    feature: str
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms: float

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


def summarize_with_budget(transcript: str) -> CallResult:
    # --- YOUR CODE HERE ---
    # 1. Record start time with time.time()
    # 2. Call client.chat.completions.create() (same prompt as Step 1)
    # 3. Calculate latency_ms = (time.time() - start) * 1000
    # 4. Return CallResult with feature='summarize', answer, input_tokens=resp.usage.prompt_tokens,
    #    output_tokens=resp.usage.completion_tokens, latency_ms
    pass


def extract_with_budget(invoice_text: str) -> CallResult:
    # --- YOUR CODE HERE ---
    pass


def answer_with_budget(question: str, context: str) -> CallResult:
    # --- YOUR CODE HERE ---
    pass


results = [
    summarize_with_budget(TRANSCRIPT),
    extract_with_budget(INVOICE_TEXT),
    answer_with_budget('Can I get a refund after 45 days?', POLICY_CHUNKS),
]

print(f'{'Feature':<12}  {'Input':>8}  {'Output':>8}  {'Total':>8}  {'Latency':>10}')
print('-' * 56)
for r in results:
    if r:
        print(f'{r.feature:<12}  {r.input_tokens:>8}  {r.output_tokens:>8}  {r.total_tokens:>8}  {r.latency_ms:>8.0f}ms')

<details>
<summary>📖 Reference solution (expand after you've tried)</summary>

```python
def summarize_with_budget(transcript: str) -> CallResult:
    start = time.time()
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Summarize this support transcript in 2-3 sentences for the handoff agent.'},
            {'role': 'user', 'content': transcript},
        ],
    )
    return CallResult(
        feature='summarize',
        answer=resp.choices[0].message.content,
        input_tokens=resp.usage.prompt_tokens,
        output_tokens=resp.usage.completion_tokens,
        latency_ms=(time.time() - start) * 1000,
    )
```

Apply the same pattern to `extract_with_budget` and `answer_with_budget`.
</details>

## 🗂️ Step 3: The Budget Worksheet

Now that you can see the numbers, classify each call by **token shape**.

For each feature, decide:
- **Dominant bucket**: Is it input-heavy, output-heavy, or balanced?
- **Latency driver**: Prefill (long input) or decode (long output)?
- **Cost lever**: Where would you cut first if the feature came in over budget?

The cell below prints the worksheet framework. Fill in your answers by editing the `budget` dict for each feature.

In [ ]:
worksheets = [
    {
        'feature': 'summarize',
        'dominant_bucket': '???',       # 'input-heavy' | 'output-heavy' | 'balanced'
        'latency_driver': '???',        # 'prefill' | 'decode'
        'first_cost_lever': '???',      # e.g. 'chunk the transcript (map-reduce)'
    },
    {
        'feature': 'extract',
        'dominant_bucket': '???',
        'latency_driver': '???',
        'first_cost_lever': '???',
    },
    {
        'feature': 'rag-answer',
        'dominant_bucket': '???',
        'latency_driver': '???',
        'first_cost_lever': '???',
    },
]

for w in worksheets:
    print(f"\n{'─'*50}")
    print(f"Feature:          {w['feature']}")
    print(f"Dominant bucket:  {w['dominant_bucket']}")
    print(f"Latency driver:   {w['latency_driver']}")
    print(f"First cost lever: {w['first_cost_lever']}")

<details>
<summary>📖 Expected answers (expand after you've filled in yours)</summary>

| Feature | Dominant bucket | Latency driver | First cost lever |
|---|---|---|---|
| summarize | **input-heavy** — long transcript, short summary | **prefill** — the big transcript dominates the prefill pass | Chunk transcript → staged summaries (map-reduce) |
| extract | **balanced / output-heavy** — short invoice, JSON schema output | **decode** — structured JSON output is generated token by token | Lean the schema (shorter field names, drop optional fields) |
| rag-answer | **input-heavy** — policy chunks dominate input | **prefill** — retrieved context is the biggest input line | Retrieve fewer/better chunks; avoid over-retrieval |

**Rule**: Identify the dominant bucket *before* you pull any optimization lever.
</details>

## 🔥 Step 4: The Expensive Feature

Here is a real trace from the production support assistant:

```
System prompt:    900 tokens   (same on every request)
Chat history:   4,000 tokens
Retrieved docs: 6,000 tokens
Tool schemas:   1,500 tokens   (same on every request)
Output:         1,200 tokens

Latency target: 3s    Actual latency: 9s
Cost target:    $0.05  Actual cost: ~$0.20 (4x over)
```

**Your task:**
1. Simulate this trace by constructing a large prompt with the right token counts
2. Identify which buckets are cacheable vs. dynamic
3. Propose the first fix — not "use a bigger context window"

Run the cell as-is first to see the baseline cost. Then implement the caching calculation.

In [ ]:
# Approximate token costs at gpt-4o-mini rates (Jan 2025):
# Input:  $0.150 per 1M tokens
# Output: $0.600 per 1M tokens
# Cached: $0.075 per 1M tokens (50% discount)
INPUT_COST_PER_TOKEN  = 0.150 / 1_000_000
OUTPUT_COST_PER_TOKEN = 0.600 / 1_000_000
CACHED_COST_PER_TOKEN = 0.075 / 1_000_000  # prompt cache hit

# Trace buckets
system_prompt_tokens  = 900    # stable — same on every request
chat_history_tokens   = 4_000  # dynamic — grows over conversation
retrieved_doc_tokens  = 6_000  # dynamic — changes per query
tool_schema_tokens    = 1_500  # stable — same on every request
output_tokens         = 1_200

total_input = system_prompt_tokens + chat_history_tokens + retrieved_doc_tokens + tool_schema_tokens
baseline_cost = (total_input * INPUT_COST_PER_TOKEN) + (output_tokens * OUTPUT_COST_PER_TOKEN)

print('=== Baseline Trace ===')
print(f'  System prompt:  {system_prompt_tokens:>6,} tokens  (stable)')
print(f'  Chat history:   {chat_history_tokens:>6,} tokens  (dynamic)')
print(f'  Retrieved docs: {retrieved_doc_tokens:>6,} tokens  (dynamic)')
print(f'  Tool schemas:   {tool_schema_tokens:>6,} tokens  (stable)')
print(f'  Output:         {output_tokens:>6,} tokens')
print(f'  Total input:    {total_input:>6,} tokens')
print(f'  Baseline cost:  ${baseline_cost:.4f} per request')

print()
print('--- Your analysis ---')
# Question 1: Which tokens are cacheable (stable prefix)?
cacheable_tokens = 0       # ← fill this in: system_prompt_tokens + tool_schema_tokens

# Question 2: What does the cost look like with the stable prefix cached?
cached_input     = total_input - cacheable_tokens   # dynamic tokens, full price
cost_with_cache  = (
    (cacheable_tokens * CACHED_COST_PER_TOKEN) +
    (cached_input * INPUT_COST_PER_TOKEN) +
    (output_tokens * OUTPUT_COST_PER_TOKEN)
)

# Question 3: What's the biggest single lever for latency?
# Hint: which dynamic bucket dominates the input prefill pass?
biggest_dynamic_bucket = '???'   # ← name it

print(f'  Cacheable tokens:        {cacheable_tokens:>6,}')
print(f'  Cost with cache:         ${cost_with_cache:.4f}')
print(f'  Saving vs baseline:      ${baseline_cost - cost_with_cache:.4f}')
print(f'  Biggest dynamic bucket:  {biggest_dynamic_bucket}')

<details>
<summary>📖 Reference answers</summary>

```python
cacheable_tokens = system_prompt_tokens + tool_schema_tokens  # 2,400 tokens
biggest_dynamic_bucket = 'retrieved_docs'  # 6,000 tokens — the largest dynamic input line
```

**What this tells you:**
- Caching the 2,400-token stable prefix saves ~50% on those tokens every request
- The 6,000-token retrieved context is the dominant latency driver (prefill-heavy)
- The fix order: **cache stable prefix first**, then **cut retrieved docs** (retrieve fewer/better chunks), then **compact history** (rolling summary)
- "Use a bigger context window" is not a fix — it doesn't change the prefill cost
</details>

## 📋 Step 5: Token Metadata in Eval Records

A quality score that says "the answer was good" is incomplete evidence.
A production eval record needs the **token shape** sitting next to the quality signal.

**Your task:** Build a `EvalRecord` dataclass and a wrapper that runs a support answer while capturing both quality signals *and* runtime metadata. Then run it against two test cases.

In [ ]:
@dataclass
class EvalRecord:
    query: str
    expected_answer_contains: str   # simple ground-truth check
    # Quality signals
    answer: str = ''
    quality_pass: bool = False
    # Runtime metadata — these are the fields a vanity-metric eval omits
    input_tokens: int = 0
    output_tokens: int = 0
    latency_ms: float = 0.0
    estimated_cost_usd: float = 0.0


def run_eval(query: str, context: str, expected_contains: str) -> EvalRecord:
    record = EvalRecord(query=query, expected_answer_contains=expected_contains)

    # --- YOUR CODE HERE ---
    # 1. Time the API call
    # 2. Call client.chat.completions.create() with the policy-answer prompt
    # 3. Fill in record.answer, record.input_tokens, record.output_tokens, record.latency_ms
    # 4. Set record.estimated_cost_usd using INPUT_COST_PER_TOKEN / OUTPUT_COST_PER_TOKEN
    # 5. Set record.quality_pass = expected_contains.lower() in record.answer.lower()

    return record


gold_set = [
    {
        'query': 'Can I get a full refund after 45 days?',
        'expected_contains': 'prorated',
    },
    {
        'query': 'What happens to my service if I cancel today?',
        'expected_contains': 'billing period',
    },
]

records = [run_eval(q['query'], POLICY_CHUNKS, q['expected_contains']) for q in gold_set]

print(f"{'Query':<45}  {'Pass':>6}  {'In':>6}  {'Out':>6}  {'ms':>6}  {'Cost':>8}")
print('-' * 82)
for r in records:
    if r.answer:
        print(f"{r.query[:44]:<45}  {str(r.quality_pass):>6}  {r.input_tokens:>6}  {r.output_tokens:>6}  {r.latency_ms:>6.0f}  ${r.estimated_cost_usd:.5f}")

print()
print('Now you can ask: which passing answers were the cheapest? Which were slowest?')
print('That is the difference between a demo eval and a production eval.')

<details>
<summary>📖 Reference solution</summary>

```python
def run_eval(query: str, context: str, expected_contains: str) -> EvalRecord:
    record = EvalRecord(query=query, expected_answer_contains=expected_contains)
    start = time.time()
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Answer the customer question using only the provided policy context.'},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'},
        ],
    )
    record.latency_ms = (time.time() - start) * 1000
    record.answer = resp.choices[0].message.content
    record.input_tokens = resp.usage.prompt_tokens
    record.output_tokens = resp.usage.completion_tokens
    record.estimated_cost_usd = (
        record.input_tokens * INPUT_COST_PER_TOKEN +
        record.output_tokens * OUTPUT_COST_PER_TOKEN
    )
    record.quality_pass = expected_contains.lower() in record.answer.lower()
    return record
```
</details>

## ✅ What You Built

| Concept | What you practiced |
|---|---|
| Token shapes | Measured three task types and classified their dominant buckets |
| Budget worksheet | Mapped input, output, latency driver, and first cost lever per task |
| Caching opportunity | Identified the stable prefix in a production trace |
| Eval metadata | Added token counts, latency, and cost alongside quality signals |

### What this changes in your workflow

Before this lab, "it's slow and expensive" was a vibe. Now you have a vocabulary:

- **Input-heavy** → fix is in the retrieval or chunking layer
- **Output-heavy** → fix is in the output contract
- **Stable prefix** → fix is caching
- **Growing history** → fix is compaction

Pull the lever that matches the bucket. Not a generic "shorten the prompt."

## 📱 Return to Synapse

Scan the QR code below to jump back to the app and continue to Phase 4.

In [ ]:
try:
    import qrcode
    from IPython.display import display
except ImportError:
    !pip install -q qrcode[pil]
    import qrcode
    from IPython.display import display

NEXT_MODULE_URL = 'synapse://learn/04-embeddings-semantic-search'

qr = qrcode.QRCode(version=1, box_size=10, border=5)
qr.add_data(NEXT_MODULE_URL)
qr.make(fit=True)
img = qr.make_image(fill_color='black', back_color='white')
display(img)